# Notebook 3 — KPI Summary for Tableau

**Goal:** Create a small set of summary tables for the Tableau dashboard. Preparing these tables in Python keeps the Tableau workbook simple and makes the calculations easier to manage.

The main KPIs used in this project are:

* Occupancy rate
* Average nightly price
* Review score
* Availability (the number of days a listing is available for booking)
* Superhost percentage (the share of listings managed by a superhost)

**Steps in this notebook:**

1. Load the clean data
2. Build the overall KPI summary
3. Build the district summary table
4. Build the room type summary table
5. Build the superhost summary table
6. Save the Tableau extract files


## Step 1 — Load the clean data

I load the cleaned listings dataset and recreate the same `priced` subset used in Notebook 2, which includes listings with prices between 10 and 1,000 euros. Applying the same filter here keeps the price metrics consistent throughout the project.


In [13]:
import pandas as pd

listings = pd.read_csv("../data/clean/listings_clean.csv")
print("Listings loaded:", listings.shape)

# Apply the same price filter used in Notebook 2.
priced = listings[listings["has_price"]].copy()
priced = priced[(priced["price"] >= 10) & (priced["price"] <= 1000)]
print("Priced listings for price KPIs:", len(priced))

Listings loaded: (14274, 28)
Priced listings for price KPIs: 9220


## Step 2 — Build the overall KPI summary

This table contains the overall metrics for the Berlin market and provides the values shown in the KPI cards at the top of the dashboard. I calculate the superhost percentage first and then combine all the metrics into a single row.


In [14]:
# Superhost percentage: the share of listings managed by a superhost.
superhost_count = (listings["host_is_superhost"] == "Yes").sum()
superhost_pct = round(superhost_count / len(listings) * 100, 1)

# Combine the overall metrics into a one-row table.
overall_kpis = pd.DataFrame([{
    "total_listings": len(listings),
    "avg_nightly_price": round(priced["price"].mean(), 0),
    "avg_occupancy_rate": round(listings["occupancy_rate"].mean(), 1),
    "avg_review_score": round(listings["review_scores_rating"].mean(), 2),
    "avg_availability_365": round(listings["availability_365"].mean(), 0),
    "superhost_percentage": superhost_pct,
}])

overall_kpis

,total_listings,avg_nightly_price,avg_occupancy_rate,avg_review_score,avg_availability_365,superhost_percentage
0,14274,129.0,21.2,4.75,147.0,24.7


## Step 3 — Build the district summary table

This table contains one row per Berlin district, with the main KPIs shown side by side. It is the most important summary table for the dashboard because it brings together price, occupancy, and review scores in one place.

To keep the process easy to follow, I calculate each metric separately and then combine the results into a single table.


In [15]:
# Number of listings in each district.
listings_per_district = listings.groupby("neighbourhood_group_cleansed")["id"].count()

# Average occupancy rate in each district.
occupancy_per_district = listings.groupby("neighbourhood_group_cleansed")["occupancy_rate"].mean()

# Average review score in each district.
score_per_district = listings.groupby("neighbourhood_group_cleansed")["review_scores_rating"].mean()

# Average availability (open days per year) in each district.
availability_per_district = listings.groupby("neighbourhood_group_cleansed")["availability_365"].mean()

# Average price in each district, from the priced subset.
price_per_district = priced.groupby("neighbourhood_group_cleansed")["price"].mean()

For each district, I calculate the superhost percentage by dividing the number of superhost listings by the total number of listings.


In [16]:
superhosts = listings[listings["host_is_superhost"] == "Yes"]
superhosts_per_district = superhosts.groupby("neighbourhood_group_cleansed")["id"].count()
superhost_pct_per_district = (superhosts_per_district / listings_per_district) * 100

I then combine all six metrics into a single table, rename the district column for clarity, and sort the results by occupancy so that the highest-performing districts appear first.


In [17]:
district_summary = pd.DataFrame({
    "total_listings": listings_per_district,
    "avg_occupancy_rate": occupancy_per_district.round(1),
    "avg_review_score": score_per_district.round(2),
    "avg_availability_365": availability_per_district.round(0),
    "avg_nightly_price": price_per_district.round(0),
    "superhost_percentage": superhost_pct_per_district.round(1),
})

# # Convert the district index into a regular column and rename it.

district_summary = district_summary.reset_index()
district_summary = district_summary.rename(
    columns={"neighbourhood_group_cleansed": "district"}
)
district_summary = district_summary.sort_values("avg_occupancy_rate",
                                                 ascending=False)
district_summary

,district,total_listings,avg_occupancy_rate,avg_review_score,avg_availability_365,avg_nightly_price,superhost_percentage
10,Tempelhof - Schöneberg,952,23.1,4.75,142.0,118.0,25.3
6,Pankow,2164,23.1,4.77,150.0,136.0,26.3
4,Mitte,3102,22.8,4.73,154.0,153.0,28.6
3,Marzahn - Hellersdorf,121,22.2,4.78,186.0,97.0,21.5
1,Friedrichshain-Kreuzberg,3108,21.8,4.77,132.0,125.0,23.3
7,Reinickendorf,201,21.7,4.75,186.0,109.0,24.4
2,Lichtenberg,413,20.8,4.73,141.0,116.0,23.5
0,Charlottenburg-Wilm.,1529,19.6,4.71,181.0,130.0,25.0
9,Steglitz - Zehlendorf,388,19.1,4.78,175.0,105.0,24.0
11,Treptow - Köpenick,632,18.0,4.77,169.0,109.0,21.8


## Step 4 — Build the room type summary table

This table contains one row per room type, showing the number of listings and their average occupancy, review score, and price. It helps answer the second project question by showing how these measures vary across room types.

I build it in the same way as the district table by calculating each metric separately and then combining the results into a single table.


In [18]:
# Calculate the count and average metrics for each room type.
listings_per_room = listings.groupby("room_type")["id"].count()
occupancy_per_room = listings.groupby("room_type")["occupancy_rate"].mean()
score_per_room = listings.groupby("room_type")["review_scores_rating"].mean()
price_per_room = priced.groupby("room_type")["price"].mean()

# Combine the metrics into a single table.
room_summary = pd.DataFrame({
    "total_listings": listings_per_room,
    "avg_occupancy_rate": occupancy_per_room.round(1),
    "avg_review_score": score_per_room.round(2),
    "avg_nightly_price": price_per_room.round(0),
})

# Convert the room type index into a regular column and sort by price.
room_summary = room_summary.reset_index()
room_summary = room_summary.sort_values("avg_nightly_price", ascending=False)
room_summary

,room_type,total_listings,avg_occupancy_rate,avg_review_score,avg_nightly_price
1,Hotel room,110,23.5,4.67,211.0
0,Entire home/apt,9663,22.3,4.76,144.0
2,Private room,4399,18.2,4.75,86.0
3,Shared room,102,39.1,4.55,48.0


## Step 5 — Build the superhost summary table

This table compares the host groups (superhost, non-superhost, and the small unknown group) using listing count, occupancy, review score, and price. It helps answer the third project question by showing how these groups differ across the main metrics.

As before, I calculate each metric separately and then combine the results into a single table.


In [19]:
# Calculate the count and average metrics for each host group.
listings_per_host = listings.groupby("host_is_superhost")["id"].count()
occupancy_per_host = listings.groupby("host_is_superhost")["occupancy_rate"].mean()
score_per_host = listings.groupby("host_is_superhost")["review_scores_rating"].mean()
price_per_host = priced.groupby("host_is_superhost")["price"].mean()

# Combine the metrics into a single table.
superhost_summary = pd.DataFrame({
    "total_listings": listings_per_host,
    "avg_occupancy_rate": occupancy_per_host.round(1),
    "avg_review_score": score_per_host.round(2),
    "avg_nightly_price": price_per_host.round(0),
})

# Convert the host group index into a regular column.
superhost_summary = superhost_summary.reset_index()
superhost_summary

,host_is_superhost,total_listings,avg_occupancy_rate,avg_review_score,avg_nightly_price
0,No,10578,13.0,4.71,124.0
1,Unknown,164,44.4,4.74,138.0
2,Yes,3532,44.6,4.86,140.0


## Step 6 — Save the Tableau extract files

I save each summary table as a separate CSV file in the `tableau` folder. These smaller files are easy to connect to Tableau and keep the dashboard structure simple.

I also save a trimmed version of the listings table containing only the columns needed for the map view, so Tableau does not have to load unnecessary columns.


In [20]:
tableau_path = "../tableau/"

overall_kpis.to_csv(tableau_path + "kpi_overall.csv", index=False)
district_summary.to_csv(tableau_path + "kpi_by_district.csv", index=False)
room_summary.to_csv(tableau_path + "kpi_by_room_type.csv", index=False)
superhost_summary.to_csv(tableau_path + "kpi_by_superhost.csv", index=False)

# # Keep only the columns needed for the Tableau map view.
columns_for_map = [
    "id", "neighbourhood_group_cleansed", "neighbourhood_cleansed",
    "latitude", "longitude", "room_type", "host_is_superhost",
    "price", "occupancy_rate", "review_scores_rating",
    "availability_365", "has_price",
]
listings_extract = listings[columns_for_map]
listings_extract.to_csv(tableau_path + "listings_for_tableau.csv", index=False)

print("Saved Tableau extract files:")
print("- kpi_overall.csv")
print("- kpi_by_district.csv")
print("- kpi_by_room_type.csv")
print("- kpi_by_superhost.csv")
print("- listings_for_tableau.csv")

Saved Tableau extract files:
- kpi_overall.csv
- kpi_by_district.csv
- kpi_by_room_type.csv
- kpi_by_superhost.csv
- listings_for_tableau.csv


## Insight summary

The data is now organised into five small summary tables:

* **kpi_overall** — overall metrics for the KPI cards.
* **kpi_by_district** — a comparison of all 12 Berlin districts.
* **kpi_by_room_type** — price and performance by room type.
* **kpi_by_superhost** — a comparison of superhosts and non-superhosts.
* **listings_for_tableau** — a trimmed listing-level file for the map view.

## Business recommendation

These tables keep the Tableau dashboard simple and efficient. Performing the calculations in Python makes them easier to check and maintain, allowing Tableau to focus on displaying the results.

The next step is to build the dashboard using the files in the `tableau` folder.
